# Train the BERT model for sentiment analysis

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!pip install -q transformers

In [3]:
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

In [4]:
import torch.utils.data as data
import torch.optim as optim
import torch.nn as nn

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [6]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_scheduler

## Prepare dataset

In [7]:
!ls /content/gdrive/MyDrive

'Colab Notebooks'


In [8]:
df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base.csv')
df = df[['ID', 'BODY', 'SENTIMENT', 'PUBLISHED_ON']]

df.loc[df['SENTIMENT'] == 'POSITIVE', ['SENTIMENT']] = 1
df.loc[df['SENTIMENT'] == 'NEGATIVE', ['SENTIMENT']] = 2
df.loc[df['SENTIMENT'] == 'NEUTRAL', ['SENTIMENT']] = 0

df = df.sample(frac=1, random_state=52).reset_index()
df.dropna(inplace=True)

In [ ]:
# filtered_df = df.loc[df['BODY'].str.len() <= 512]
# print(len(filtered_df)/len(df))
# df = filtered_df

In [9]:
df = df.loc[:, ['ID', 'BODY', 'SENTIMENT', 'PUBLISHED_ON']]
for i in df.values:
  print(i)
  break

[51961637
 'Solana founder Anatoly Yakovenko forecasts a “50/50” chance of a quantum computing breakthrough by 2030, and says the Bitcoin community must “speed things up.”'
 0 1758262009]


In [10]:
list_dataframes = []

for i in df.values:
  if len(i[1]) > 512:
    bodies = []
    art = i[1]
    ln = len(art)
    while ln > 512:
      bodies.append(art[:512])
      art = art[512:]
      ln -= 512
    bodies.append(art)

    curr_df = pd.DataFrame(columns=['ID', 'BODY', 'SENTIMENT', 'PUBLISHED_ON'])
    curr_df['ID'] = [i[0]] * len(bodies)
    curr_df['BODY'] = bodies
    curr_df['SENTIMENT'] = [i[2]] * len(bodies)
    curr_df['PUBLISHED_ON'] = [i[3]] * len(bodies)

    list_dataframes.append(curr_df)
  else:
    list_dataframes.append(pd.DataFrame([i], columns=['ID', 'BODY', 'SENTIMENT', 'PUBLISHED_ON']))

df = pd.concat(list_dataframes)

In [11]:
len(df.loc[df['BODY'].str.len() <= 512]), len(df)

(658835, 658835)

In [13]:
df.to_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/data_coindesk_base_large.csv', index=False)

In [14]:
X = df['BODY']
y = df['SENTIMENT']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5)

X_train = X_train.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)

y_test = y_test.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)


## Download pretrained models

In [15]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

label2id = {'NEGATIVE': 0, 'POSITIVE': 1, 'NEUTRAL': 2}
id2label = {0: 'NEGATIVE', 1: 'POSITIVE', 2: 'NEUTRAL'}

model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=id2label,
    label2id=label2id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## DistilBERT architecture

In [16]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [17]:
model.distilbert.parameters()

<generator object Module.parameters at 0x7a3a5448a340>

In [18]:
next(iter(model.distilbert.parameters())), next(iter(model.distilbert.parameters())).requires_grad

(Parameter containing:
 tensor([[-0.0166, -0.0666, -0.0163,  ..., -0.0200, -0.0514, -0.0264],
         [-0.0132, -0.0673, -0.0161,  ..., -0.0227, -0.0554, -0.0260],
         [-0.0176, -0.0709, -0.0144,  ..., -0.0246, -0.0596, -0.0232],
         ...,
         [-0.0231, -0.0588, -0.0105,  ..., -0.0195, -0.0262, -0.0212],
         [-0.0490, -0.0561, -0.0047,  ..., -0.0107, -0.0180, -0.0219],
         [-0.0065, -0.0915, -0.0025,  ..., -0.0151, -0.0504,  0.0460]],
        device='cuda:0', requires_grad=True),
 True)

## Tokenizer architecture

In [19]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [20]:
import inspect
inspect.signature(tokenizer.__call__)

<Signature (text: Union[str, list[str], list[list[str]], NoneType] = None, text_pair: Union[str, list[str], list[list[str]], NoneType] = None, text_target: Union[str, list[str], list[list[str]], NoneType] = None, text_pair_target: Union[str, list[str], list[list[str]], NoneType] = None, add_special_tokens: bool = True, padding: Union[bool, str, transformers.utils.generic.PaddingStrategy] = False, truncation: Union[bool, str, transformers.tokenization_utils_base.TruncationStrategy, NoneType] = None, max_length: Optional[int] = None, stride: int = 0, is_split_into_words: bool = False, pad_to_multiple_of: Optional[int] = None, padding_side: Optional[str] = None, return_tensors: Union[str, transformers.utils.generic.TensorType, NoneType] = None, return_token_type_ids: Optional[bool] = None, return_attention_mask: Optional[bool] = None, return_overflowing_tokens: bool = False, return_special_tokens_mask: bool = False, return_offsets_mapping: bool = False, return_length: bool = False, verbos

## unfreeze DistilBERT weights, unfreeze Classifier weights

In [21]:
for param in model.distilbert.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

## Processing Dataset

In [ ]:
list(tokenizer(list(X_train.values)[:2]).keys())

['input_ids', 'attention_mask']

## Create SentimentDataset

In [ ]:
class SentimentDataset:
    def __init__(self, articles, targets, tokenizer):
        '''
        args:
            articles: pandas.Series
            targets: pandas.Series
            tokenizer: AutoTokenizer from Hugging Face
        '''

        self.tokenizer = tokenizer
        self.articles = list(articles.values)
        self.targets = targets
        self.articles_ids = self.tokenize_func(self.articles)


    def __getitem__(self, indx):
        '''
        get an item from dataset by it's index
        '''
        article_ids = self.articles_ids['input_ids'][indx]
        attention_mask = self.articles_ids['attention_mask'][indx]
        target = self.targets[indx]

        return article_ids, attention_mask, target

    def __len__(self):
        return len(self.articles)


    def tokenize_func(self, arts):
        '''tokenize & convert to ids
            returns input_ids, attention_mask
            (token_type_ids won't returns, we are working with 1 consisted text)
        '''
        return self.tokenizer(
            arts,
            padding='max_length',
            max_length=512,
            padding_side='right',
            truncation=True,
            return_token_type_ids='token_type_ids' in self.tokenizer.model_input_names)

## Collate batches function

In [ ]:
def collate_batch(batch, device=device):
    arts_ids, att_masks, targets = [], [], []

    for a, m, t in batch:
        arts_ids.append(a)
        att_masks.append(m)
        targets.append(t)

    article_ids = torch.tensor(arts_ids, dtype=torch.long).to(device)
    attention_mask = torch.tensor(att_masks, dtype=torch.long).to(device)
    target = torch.tensor(targets, dtype=torch.long).to(device)

    return {
        'text_ids': article_ids,
        'attention_mask': attention_mask,
        'target': target}

## Datasets & Dataloaders

In [ ]:
train_ds = SentimentDataset(X_train, y_train, tokenizer)
val_ds = SentimentDataset(X_val, y_val, tokenizer)
test_ds = SentimentDataset(X_test, y_test, tokenizer)

train_dl = data.DataLoader(train_ds, 77, shuffle=False, collate_fn=collate_batch)
val_dl = data.DataLoader(val_ds, 77, shuffle=False, collate_fn=collate_batch)
test_dl = data.DataLoader(test_ds, 77, shuffle=False, collate_fn=collate_batch)

## Evaluate function


In [ ]:
def evaluate(model, dl):
    model.eval()

    loss_fn = nn.CrossEntropyLoss()

    losses = []
    preds = []
    targets = []

    with torch.no_grad():
        for batch in dl:

            predict = model(
                input_ids=batch['text_ids'],
                attention_mask=batch['attention_mask'])
            loss = loss_fn(predict.logits, batch['target'])

            targets.append(batch['target'])
            losses.append(loss.item())
            preds.append(predict.logits.argmax(dim=-1))

    preds = torch.cat(preds)
    targets = torch.cat(targets)

    mean_loss = np.mean(losses)
    acc = (preds == targets).float().mean().item()

    model.train()

    return mean_loss, acc

## Elementary Eval

In [ ]:
val_loss, val_acc = evaluate(model, val_dl)
print('Start validation evaluation:')
print(f'loss: {val_loss}', f'accuracy: {val_acc}')

Start validation evaluation:
loss: 1.095051362210472 accuracy: 0.2764042913913727


## Training

In [ ]:
EPOCHS = 10
LR = 0.0001

optimizer = optim.AdamW(params=model.parameters(), lr=LR)
loss_func = nn.CrossEntropyLoss()
num_training_steps = EPOCHS * len(train_dl)
scheduler = get_scheduler(
    name='linear', optimizer=optimizer,
    num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
model_path = '/content/gdrive/MyDrive/Colab Notebooks/course/notebooks/transformers/model_distilbert_v0'
progress_bar = tqdm(range(num_training_steps))

count_early = 0
best_loss_val = float('inf')
model.train()

accuracies = {'train': [], 'val': []}
losses = {'train': [], 'val': []}

for epoch in range(EPOCHS):
    curr_loss = []
    preds = []
    targets = []

    # dataloader 1
    for i, batch in enumerate(tqdm(train_dl, desc=f'DL: 1 \nEpoch {epoch+1}/{EPOCHS}')):
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch['text_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['target'])
        loss = loss_func(outputs.logits, batch['target'])

        loss.backward()
        optimizer.step()
        scheduler.step()
        progress_bar.update(1)

        curr_loss.append(loss.item())
        preds.append(outputs.logits.argmax(dim=-1))
        targets.append(batch['target'])


    preds = torch.cat(preds)
    targets = torch.cat(targets)

    epoch_acc = (preds == targets).float().mean().item()
    epoch_loss = np.mean(curr_loss)

    # EVALUATION
    mean_loss_val, acc_val = evaluate(model, val_dl)

    losses['train'].append(epoch_loss)
    accuracies['train'].append(epoch_acc)

    losses['val'].append(mean_loss_val)
    accuracies['val'].append(acc_val)

    print(f'Loss train/val in this epoch: {epoch_loss}/{mean_loss_val}')
    print(f'Accuracy train/val in this epoch: {epoch_acc}/{acc_val}')

    if mean_loss_val < best_loss_val:
        best_loss_val = mean_loss_val
        count_early = 0

        st = model.state_dict()
        torch.save(st, model_path)
    else:
        count_early += 1

    if count_early > 1:
        print('Early stopping!')
        break

  0%|          | 0/11900 [00:00<?, ?it/s]

DL: 1 
Epoch 1/10:   0%|          | 0/1190 [00:00<?, ?it/s]

Loss train/val in this epoch: 0.5243875307445767/0.4640237859031498
Accuracy train/val in this epoch: 0.7804909944534302/0.8017821311950684


DL: 1 
Epoch 2/10:   0%|          | 0/1190 [00:00<?, ?it/s]

Loss train/val in this epoch: 0.3625412794597009/0.5150917120427894
Accuracy train/val in this epoch: 0.8542285561561584/0.7995107769966125


DL: 1 
Epoch 3/10:   0%|          | 0/1190 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Explain some model logic

In [ ]:
for batch in test_dl:
    outputs = model(input_ids=batch['text_ids'], attention_mask=batch['attention_mask'])
    print(outputs.logits, outputs.logits.shape)
    print(dir(outputs))
    break

In [ ]:
torch.tensor([[1, 2, 3], [2, 4, 5]]).argmax(dim=-1)